# HendiCode - Treinamento no Google Colab

Este notebook treina o modelo HendiCode (1.5B parâmetros) na GPU T4 do Google Colab.

## Pré-requisitos
1. Ter os arquivos no Google Drive:
   - `hendicode_modelo/dados_processados/tudo.bin`
   - `hendicode_modelo/models/hendicode/tokenizer.json`
   - `hendicode_modelo/models/hendicode/tokenizer_config.json`
2. Uma GPU habilitada no Colab (Runtime → Change runtime type → T4 GPU)

## Instruções
- Execute cada célula em ordem (Ctrl+F9 executa tudo)
- O treino salva checkpoints no Drive a cada 500 steps
- Ao final, o modelo final estará em `hendicode_modelo/models/hendicode/pesos_finais.pt`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado!')
!ls '/content/drive/MyDrive/hendicode_modelo/'

In [ ]:
!pip install -q torch tokenizers tqdm
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
import os, shutil

ORIGEM = '/content/drive/MyDrive/hendicode_modelo'

# Cria estrutura de pastas
os.makedirs('dados_processados', exist_ok=True)
os.makedirs('models/hendicode', exist_ok=True)

# Copia dados
if os.path.exists(f'{ORIGEM}/dados_processados/tudo.bin'):
    shutil.copy2(f'{ORIGEM}/dados_processados/tudo.bin', 'dados_processados/tudo.bin')
    print(f'Dados: {os.path.getsize("dados_processados/tudo.bin") / 1e6:.1f} MB')
else:
    # Procura .bin alternativos
    bins = [f for f in os.listdir(f'{ORIGEM}/dados_processados/') if f.endswith('.bin')] if os.path.exists(f'{ORIGEM}/dados_processados/') else []
    if bins:
        for b in bins:
            shutil.copy2(f'{ORIGEM}/dados_processados/{b}', f'dados_processados/{b}')
            print(f'Copiado: {b}')
    else:
        raise FileNotFoundError('Nenhum arquivo .bin encontrado no Drive!')

# Copia tokenizer
for f in ['tokenizer.json', 'tokenizer_config.json']:
    src = f'{ORIGEM}/models/hendicode/{f}'
    if os.path.exists(src):
        shutil.copy2(src, f'models/hendicode/{f}')
        print(f'Tokenizer: {f}')

print('\nArquivos prontos!')

In [ ]:
%%writefile modelo_original.py
import torch
import torch.nn as nn
import torch.nn.functional as F

default_config = {
    "vocab_size": 32768,
    "hidden_size": 2048,
    "intermediate_size": 8192,
    "num_hidden_layers": 24,
    "num_attention_heads": 16,
    "num_key_value_heads": 4,
    "max_position_embeddings": 2048,
    "rms_norm_eps": 1e-6,
    "rope_theta": 10000.0,
    "initializer_range": 0.02,
    "use_cache": True,
    "pad_token_id": 0,
    "bos_token_id": 1,
    "eos_token_id": 2,
}


def precompute_rope_frequencies(head_dim, max_position_embeddings, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
    t = torch.arange(max_position_embeddings, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rotary_emb(x, cos, sin):
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    cos = cos[:x.shape[1]].unsqueeze(0).unsqueeze(2)
    sin = sin[:x.shape[1]].unsqueeze(0).unsqueeze(2)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config["hidden_size"]
        self.num_heads = config["num_attention_heads"]
        self.num_kv_heads = config["num_key_value_heads"]
        self.head_dim = self.hidden_size // self.num_heads
        self.num_groups = self.num_heads // self.num_kv_heads

        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

        cos, sin = precompute_rope_frequencies(
            self.head_dim, config["max_position_embeddings"], config["rope_theta"]
        )
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, attention_mask=None):
        batch, seq_len, _ = x.shape

        q = self.q_proj(x).view(batch, seq_len, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(batch, seq_len, self.num_kv_heads, self.head_dim)
        v = self.v_proj(x).view(batch, seq_len, self.num_kv_heads, self.head_dim)

        q = apply_rotary_emb(q, self.rope_cos, self.rope_sin)
        k = apply_rotary_emb(k, self.rope_cos, self.rope_sin)

        k = k.repeat_interleave(self.num_groups, dim=2)
        v = v.repeat_interleave(self.num_groups, dim=2)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        attn_output = attn_output.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        return self.o_proj(attn_output)


class SwiGLUFFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config["hidden_size"], config["intermediate_size"], bias=False)
        self.up_proj = nn.Linear(config["hidden_size"], config["intermediate_size"], bias=False)
        self.down_proj = nn.Linear(config["intermediate_size"], config["hidden_size"], bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = GroupedQueryAttention(config)
        self.feed_forward = SwiGLUFFN(config)
        self.input_norm = RMSNorm(config["hidden_size"], config["rms_norm_eps"])
        self.post_attn_norm = RMSNorm(config["hidden_size"], config["rms_norm_eps"])

    def forward(self, x):
        residual = x
        x = self.input_norm(x)
        x = self.self_attn(x)
        x = residual + x

        residual = x
        x = self.post_attn_norm(x)
        x = self.feed_forward(x)
        x = residual + x
        return x


class HendiCodeModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config["vocab_size"], config["hidden_size"])
        self.layers = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config["num_hidden_layers"])]
        )
        self.norm = RMSNorm(config["hidden_size"], config["rms_norm_eps"])
        self.lm_head = nn.Linear(config["hidden_size"], config["vocab_size"], bias=False)
        self.embed_tokens.weight = self.lm_head.weight
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=self.config["initializer_range"])
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=self.config["initializer_range"])

    def forward(self, input_ids):
        x = self.embed_tokens(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=256, temperature=0.7, top_k=50, top_p=0.9, eos_token_id=None):
        self.eval()
        if eos_token_id is None:
            eos_token_id = self.config.get("eos_token_id", 2)
        for _ in range(max_new_tokens):
            seq_len = input_ids.shape[1]
            if seq_len > self.config["max_position_embeddings"]:
                input_ids = input_ids[:, -self.config["max_position_embeddings"]:]
            logits = self.forward(input_ids)
            next_token_logits = logits[:, -1, :]
            if temperature > 0:
                next_token_logits = next_token_logits / temperature
            else:
                next_token = next_token_logits.argmax(dim=-1, keepdim=True)
                input_ids = torch.cat([input_ids, next_token], dim=-1)
                yield next_token
                if next_token.item() == eos_token_id:
                    break
                continue
            if top_k > 0:
                top_k_vals, _ = torch.topk(next_token_logits, top_k, dim=-1)
                min_top_k = top_k_vals[:, -1].unsqueeze(-1)
                next_token_logits = torch.where(
                    next_token_logits < min_top_k,
                    torch.tensor(float("-inf"), device=next_token_logits.device),
                    next_token_logits,
                )
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True, dim=-1)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = False
                indices_to_remove = sorted_indices_to_remove.scatter(-1, sorted_indices, sorted_indices_to_remove)
                next_token_logits = next_token_logits.masked_fill(indices_to_remove, float("-inf"))
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            yield next_token
            if next_token.item() == eos_token_id:
                break

print('modelo_original.py criado com sucesso!')

In [ ]:
%%writefile treinar_colab.py
import os, sys, glob, json, time, math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from modelo_original import HendiCodeModel, default_config

class TextDataset(Dataset):
    def __init__(self, data_dir="dados_processados", seq_length=1024):
        self.seq_length = seq_length
        self.data = []
        arquivos = sorted(glob.glob(os.path.join(data_dir, "*.bin")))
        if not arquivos:
            raise FileNotFoundError(f"Nenhum .bin encontrado em {data_dir}/")
        for arq in arquivos:
            with open(arq, "rb") as f:
                raw = f.read()
            tokens = [int.from_bytes(raw[i:i+4], "little") for i in range(0, len(raw), 4)]
            self.data.extend(tokens)
            nome = os.path.basename(arq)
            print(f"  {nome}: {len(tokens):,} tokens")
        self.data = torch.tensor(self.data, dtype=torch.long)
        print(f"  Total: {len(self.data):,} tokens")
    def __len__(self):
        return max(0, len(self.data) - self.seq_length)
    def __getitem__(self, idx):
        chunk = self.data[idx:idx + self.seq_length + 1]
        return chunk[:-1], chunk[1:]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

SEQ_LENGTH = 1024
BATCH_SIZE = 1
GRAD_ACCUM = 16
EPOCHS = 3
LR = 2e-4
SAVE_EVERY = 500
OUTPUT_DIR = "models/hendicode"

dataset = TextDataset(seq_length=SEQ_LENGTH)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
print(f"Batches por época: {len(loader)}")

config = dict(default_config)
config["max_position_embeddings"] = SEQ_LENGTH

model = HendiCodeModel(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros: {total_params:,}")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95))
total_steps = len(loader) * EPOCHS // GRAD_ACCUM
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR * 0.1)
scaler = torch.cuda.amp.GradScaler()

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Salva config do modelo
with open(os.path.join(OUTPUT_DIR, "model_config.json"), "w") as f:
    json.dump(config, f, indent=2)

model.train()
global_step = 0
tokens_total = 0
inicio = time.time()

print(f"\nIniciando treino: {EPOCHS} épocas, {total_steps} steps totais")
print("=" * 60)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    epoch_loss = 0.0
    epoch_tokens = 0
    epoch_inicio = time.time()

    for batch_idx, (inputs, targets) in enumerate(loader):
        inputs = inputs.to(device)
        targets = targets.to(device)

        with torch.cuda.amp.autocast():
            logits = model(inputs)
            loss = nn.CrossEntropyLoss()(logits.view(-1, config["vocab_size"]), targets.view(-1))
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        tokens_total += inputs.numel()
        epoch_tokens += inputs.numel()

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1
            epoch_loss += loss.item() * GRAD_ACCUM

            if global_step % 20 == 0:
                elapsed = time.time() - inicio
                tok_s = tokens_total / max(elapsed, 1)
                loss_val = loss.item() * GRAD_ACCUM
                lr_atual = scheduler.get_last_lr()[0]
                print(f"[Ep {epoch+1}/{EPOCHS}] Step {global_step}/{total_steps} | Loss: {loss_val:.4f} | LR: {lr_atual:.2e} | Tok/s: {tok_s:.0f}")

            if global_step % SAVE_EVERY == 0:
                caminho = os.path.join(OUTPUT_DIR, f"checkpoint_step_{global_step}.pt")
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "step": global_step,
                    "epoch": epoch,
                    "loss": loss.item() * GRAD_ACCUM,
                }, caminho)
                print(f">>> Checkpoint salvo: {caminho}")

                # Copia checkpoint para o Drive
                import subprocess
                destino = f"/content/drive/MyDrive/hendicode_modelo/{OUTPUT_DIR}/checkpoint_step_{global_step}.pt"
                os.makedirs(os.path.dirname(destino), exist_ok=True)
                subprocess.run(["cp", caminho, destino])
                print(f">>> Copiado para o Drive: {destino}")

    epoch_duracao = time.time() - epoch_inicio
    print(f"\n--- Época {epoch+1} concluída em {epoch_duracao/60:.1f} min | Loss média: {epoch_loss / max(global_step, 1):.4f} ---\n")

    # Salva checkpoint de época
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f"checkpoint_epoca_{epoch+1}.pt"))

elapsed_total = time.time() - inicio
print(f"\n{'=' * 60}")
print(f"Treino concluído em {elapsed_total/3600:.2f} horas!")
print(f"Tokens processados: {tokens_total:,}")
print(f"{'=' * 60}")

# Salva modelo final
caminho_final = os.path.join(OUTPUT_DIR, "pesos_finais.pt")
torch.save(model.state_dict(), caminho_final)
print(f"Modelo final salvo: {caminho_final}")

# Copia pro Drive
import subprocess
destino_final = f"/content/drive/MyDrive/hendicode_modelo/{OUTPUT_DIR}/pesos_finais.pt"
subprocess.run(["cp", caminho_final, destino_final])
print(f"Copiado para o Drive: {destino_final}")

In [ ]:
!python treinar_colab.py

In [ ]:
from tokenizers import Tokenizer
from modelo_original import HendiCodeModel, default_config
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Carrega tokenizer
tok = Tokenizer.from_file('models/hendicode/tokenizer.json')

# Carrega modelo
config = dict(default_config)
model = HendiCodeModel(config).to(device)
model.load_state_dict(torch.load('models/hendicode/pesos_finais.pt', map_location=device))
model.eval()

print('Modelo carregado!')

# Teste de geração
prompt = 'def fibonacci(n):'
input_ids = torch.tensor([tok.encode(prompt).ids], dtype=torch.long, device=device)

print(f'Prompt: {prompt}')
print('Gerando...')

tokens = []
for token_id in model.generate(input_ids, max_new_tokens=100):
    tokens.append(token_id.item() if hasattr(token_id, 'item') else token_id)
    # Você também pode fazer decode em tempo real se quiser

output = tok.decode(tokens)
print(f'Output: {output}')